# Proyecto IA en Ingeniería Estructural - UPC
## Modelo Supervisado Físicamente Informado
Este notebook presenta el flujo completo de entrenamiento supervisado aplicado al análisis de riesgo sísmico según la norma E.030 del RNE. Se ha estructurado para hacer visible y explicativo todo el proceso de Machine Learning, incluyendo métricas y gráficas para su evaluación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix

# Configuración visual de las gráficas
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Carga y Exploración de Datos
Cargamos el dataset generado y visualizamos una muestra de los datos y su estructura.

In [ ]:
df = pd.read_csv('dataset_e030_v2.csv')
display(df.head())
print("\nInformación del Dataset:")
df.info()

### 2. Definición de Features y Targets
Separamos las variables predictoras (características estructurales y sísmicas) de las variables objetivo (Cortantes, Deriva y Peligrosidad).

In [ ]:
feature_cols = [
    'Factor_Z', 'Factor_S', 'Periodo_Tp_s', 'Periodo_TL_s', 'Factor_Uso_U',
    'R0_X', 'R0_Y', 'Irregularidad_Altura_Ia', 'Irregularidad_Planta_Ip',
    'Factor_R_X', 'Factor_R_Y', 'N_Pisos', 'Altura_Total_H_m', 'Area_Planta_m2',
    'Peso_Total_P_Ton', 'Ratio_Area_Muros_pct', 'Periodo_Tx_s', 'Periodo_Ty_s',
    'Factor_Cx', 'Factor_Cy'
]

X = df[feature_cols]
y_vx = df['Cortante_Basal_Vx_Ton']
y_vy = df['Cortante_Basal_Vy_Ton']
y_deriva = df['Deriva_Inelastica_Max_X']
y_clf = df['Indice_Peligrosidad']

### 3. División de Datos (Train / Test)
Se separa el 80% de los datos para entrenamiento y el 20% para pruebas, estratificando por el índice de peligrosidad para mantener el balanceo de clases.

In [ ]:
X_tr, X_te, y_vx_tr, y_vx_te, y_vy_tr, y_vy_te, y_d_tr, y_d_te, y_c_tr, y_c_te = train_test_split(
    X, y_vx, y_vy, y_deriva, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"Set de entrenamiento: {X_tr.shape[0]} muestras")
print(f"Set de prueba: {X_te.shape[0]} muestras")

### 4. Etapa 1: Regresión de Demandas Físicas
Entrenamos modelos `GradientBoostingRegressor` para predecir las demandas físicas: Cortante Basal en X, Cortante Basal en Y, y la Deriva Inelástica.

In [ ]:
reg_vx = GradientBoostingRegressor(n_estimators=150, random_state=42).fit(X_tr, y_vx_tr)
reg_vy = GradientBoostingRegressor(n_estimators=150, random_state=42).fit(X_tr, y_vy_tr)
reg_d = GradientBoostingRegressor(n_estimators=150, random_state=42).fit(X_tr, y_d_tr)

pred_vx = reg_vx.predict(X_te)
pred_vy = reg_vy.predict(X_te)
pred_d = reg_d.predict(X_te)

print(f"R2 Cortante Vx: {r2_score(y_vx_te, pred_vx):.4f}")
print(f"R2 Cortante Vy: {r2_score(y_vy_te, pred_vy):.4f}")
print(f"R2 Deriva Inelástica: {r2_score(y_d_te, pred_d):.4f}")

#### 4.1 Visualización: Valores Reales vs. Predicciones
A continuación, graficamos qué tan precisas son las predicciones del modelo comparadas con los valores reales. La línea roja representa la predicción perfecta.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cortante Vx
axes[0].scatter(y_vx_te, pred_vx, alpha=0.5, color='blue')
axes[0].plot([y_vx_te.min(), y_vx_te.max()], [y_vx_te.min(), y_vx_te.max()], 'r--')
axes[0].set_title('Cortante Vx: Real vs Predicho')
axes[0].set_xlabel('Valor Real (Ton)')
axes[0].set_ylabel('Predicción (Ton)')

# Cortante Vy
axes[1].scatter(y_vy_te, pred_vy, alpha=0.5, color='green')
axes[1].plot([y_vy_te.min(), y_vy_te.max()], [y_vy_te.min(), y_vy_te.max()], 'r--')
axes[1].set_title('Cortante Vy: Real vs Predicho')
axes[1].set_xlabel('Valor Real (Ton)')

# Deriva
axes[2].scatter(y_d_te, pred_d, alpha=0.5, color='purple')
axes[2].plot([y_d_te.min(), y_d_te.max()], [y_d_te.min(), y_d_te.max()], 'r--')
axes[2].set_title('Deriva Inelástica: Real vs Predicho')
axes[2].set_xlabel('Valor Real')

plt.tight_layout()
plt.show()

### 5. Etapa 2: Clasificación de Peligrosidad Sísmica
Utilizamos un enfoque Físicamente Informado: incorporamos las predicciones de la etapa de regresión como **nuevas características (variables)** al dataset de clasificación.

In [ ]:
# Aumentar características de entrenamiento y prueba
X_tr_aug = X_tr.copy()
X_tr_aug['Pred_Vx'] = reg_vx.predict(X_tr)
X_tr_aug['Pred_Vy'] = reg_vy.predict(X_tr)
X_tr_aug['Pred_Deriva'] = reg_d.predict(X_tr)

X_te_aug = X_te.copy()
X_te_aug['Pred_Vx'] = pred_vx
X_te_aug['Pred_Vy'] = pred_vy
X_te_aug['Pred_Deriva'] = pred_d

# Entrenamiento del Clasificador
clf = GradientBoostingClassifier(n_estimators=150, random_state=42).fit(X_tr_aug, y_c_tr)
pred_c = clf.predict(X_te_aug)

print(f"Accuracy Peligrosidad Sísmica: {accuracy_score(y_c_te, pred_c)*100:.2f}%\n")
print("Reporte de Clasificación:")
print(classification_report(y_c_te, pred_c))

#### 5.1 Visualización: Matriz de Confusión e Importancia de Variables
Observamos los aciertos y errores del clasificador, y evaluamos cuáles fueron las variables más importantes para tomar la decisión.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matriz de Confusión
cm = confusion_matrix(y_c_te, pred_c)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusión - Peligrosidad')
axes[0].set_xlabel('Peligrosidad Predicha')
axes[0].set_ylabel('Peligrosidad Real')

# Importancia de Variables
importances = clf.feature_importances_
indices = np.argsort(importances)[-15:] # Top 15 variables más importantes

axes[1].barh(range(len(indices)), importances[indices], align='center', color='teal')
axes[1].set_yticks(range(len(indices)))
axes[1].set_yticklabels([X_tr_aug.columns[i] for i in indices])
axes[1].set_title('Top 15 Características Más Importantes para Clasificar Peligrosidad')

plt.tight_layout()
plt.show()